In [ ]:
pip install chromadb

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import chromadb

# 1. Load embedding model (for retrieval / Chroma)
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

# 2. Load open-source LLM from Hugging Face
#    (you can swap to another chat/instruct model)
model_name = "HuggingFaceH4/zephyr-7b-beta"   # example instruct model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# If you don't have GPU, you can drop torch_dtype & device_map
llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
device = llm.device


In [ ]:
# 3. Init ChromaDB (in-memory) and create a collection

client = chromadb.Client() 
collection = client.create_collection(
    name="rag_docs",
    metadata={"hnsw:space": "cosine"}  # cosine similarity
)

# 4. Add some documents to our "knowledge base"
#    (in practice, these would be chunks from PDFs, DB, etc.)
documents = [
    "Retrieval-Augmented Generation, or RAG, combines a retriever with a generator.",
    "RAG improves answer accuracy by grounding the LLM on external knowledge sources.",
    "Vector databases such as Chroma, FAISS, and Weaviate store embeddings for fast similarity \
    search.",
    "Text embeddings map text into a high-dimensional vector space where semantic similarity is \
    preserved.",
]

doc_embeddings = embedder.encode(documents).tolist()

collection.add(
    ids=[str(i) for i in range(len(documents))],
    documents=documents,
    embeddings=doc_embeddings,
)



In [ ]:
# 5. RAG query function

def rag_query(question: str, k: int = 2, max_new_tokens: int = 200) -> str:
    # 5.1 Embed the question
    q_embedding = embedder.encode([question]).tolist()[0]

    # 5.2 Retrieve top-k relevant docs from Chroma
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=k,
    )
    retrieved_docs = results["documents"][0]

    # 5.3 Build an instruct-style prompt using retrieved context
    context = "\n\n".join(retrieved_docs)
    prompt = (
        "You are a helpful assistant. Use ONLY the context below to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )

    # 5.4 Generate with the LLM
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    full_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Optionally, strip the prompt part if you only want the answer
    # (many instruct models just append the answer at the end)
    if "Answer:" in full_output:
        answer = full_output.split("Answer:", 1)[-1].strip()
    else:
        answer = full_output.strip()

    return answer


In [ ]:
# 6. Testing the RAG system with a question

q = "What is RAG and why is it useful?"
print("Question:", q)
ans = rag_query(q)
print("\nAnswer:\n", ans)